### Stage-1. Problem, Data, KPIs definition, and Assumptions

In a Plate Mill, steel slabs are reheated in furnaces and rolled into plates for customers.
Each customer order specifies one or more plates with required dimensions and material properties. To fulfil an order, the plant must allocate slabs that can be rolled into the requested plates.

We aim to design a production-ready system that recommends slab-to-order allocations that minimise waste while satisfying operational constraints.


#### 1. Defining Material classes (Slab and Plate)

The data model explicitly captures the key operational compatibility requirements as specified in the problem:

- **Steel grade:** Both slabs and plates carry a grade attribute. An allocation is only considered feasible when the slab grade is compatible with the plate requirement.
- **Quality specification:** Slabs and plates also contain a quality specification, ensuring that material is only allocated where the required quality is satisfied.
- **Customer-specific requirements:** Customer restrictions are associated with the order and can impose additional constraints, such as approved slab suppliers or required surface classifications.
- **Separation of concerns:** These attributes are stored in the relevant domain objects, while a separate compatibility function determines whether a particular slab–plate allocation is feasible.

In [13]:
class Slab:

    def __init__(
        self,
        slab_id,
        length,
        grade,
        quality,
        supplier,
        surface_class,
        status="available"
    ):
        self.slab_id = slab_id
        self.length = length
        self.grade = grade
        self.quality = quality
        self.supplier = supplier
        self.surface_class = surface_class
        self.status = status


class Plate:

    def __init__(
        self,
        plate_id,
        order_id,
        length,
        width,
        thickness,
        grade,
        quality
    ):
        self.plate_id = plate_id
        self.order_id = order_id
        self.length = length
        self.width = width
        self.thickness = thickness
        self.grade = grade
        self.quality = quality

class Order:

    def __init__(
        self,
        order_id,
        customer,
        plates,
        allowed_suppliers=None,
        required_surface_class=None,
        require_full_fulfilment=True
    ):
        self.order_id = order_id
        self.customer = customer
        self.plates = plates
        self.allowed_suppliers = allowed_suppliers
        self.required_surface_class = required_surface_class
        self.require_full_fulfilment = require_full_fulfilment

#### 2. Defining Compatibility for orders and Required Length Calculation

The allocation process first checks whether a slab is compatible with a plate and its customer order based on steel grade, quality specification, customer-specific restrictions, and slab inventory status.

Only compatible slab–plate pairs are considered as feasible allocation candidates. For each feasible pair, the required slab length is then calculated.

For demonstration purposes, separate rolling/yield allowances are assumed for the two steel grades. In a real-world implementation, this function would be replaced by the plant's established process model.

In [145]:
def is_compatible(slab, plate, order):

    # Steel grade
    if slab.grade != plate.grade:
        return False, "Steel grade mismatch"

    # Quality specification
    if slab.quality != plate.quality:
        return False, "Quality specification mismatch"

    # Customer-specific supplier restriction
    if order.allowed_suppliers is not None:
        if slab.supplier not in order.allowed_suppliers:
            return False, "Supplier not permitted for customer"

    # Customer-specific surface restriction
    if order.required_surface_class is not None:
        if slab.surface_class != order.required_surface_class:
            return False, "Surface class does not meet customer requirement"

    # Inventory status
    if slab.status != "Available":
        return False, f"Slab is {slab.status}"

    return True, "Compatible"


SLAB_CROSS_SECTION = 0.04  # m², assumed reference value

def calculate_required_length(slab, plate):

    if slab.grade != plate.grade:
        raise ValueError("Incompatible slab grade")

    if slab.grade == "G1":
        yield_factor = 0.94

    elif slab.grade == "G2":
        yield_factor = 0.88

    else:
        raise ValueError(
            f"Unsupported slab grade: {slab.grade}"
        )

    plate_volume = (
        plate.length *
        plate.width *
        plate.thickness
    )

    required_length = (
        plate_volume /
        (SLAB_CROSS_SECTION * yield_factor)
    )

    return required_length

#### 3. Allocation Problem and KPIs

An `Allocation` represents a feasible assignment of a customer plate to a compatible slab. The allocation also calculates the required slab length for the selected slab–plate combination.

The allocation performance is evaluated using material consumption, yield loss, slab utilisation, and order/plate fulfilment metrics. These KPIs will later be used to compare the baseline allocation approach with the SCIP optimisation model.

In [162]:
class Allocation:

    def __init__(self, slab, plate, order):

        compatible, reason = is_compatible(
            slab,
            plate,
            order
        )

        if not compatible:
            raise ValueError(
                f"Slab {slab.slab_id} is not compatible "
                f"with Plate {plate.plate_id}: {reason}"
            )

        self.slab = slab
        self.plate = plate
        self.order = order

        self.required_length = calculate_required_length(
            slab,
            plate
        )


def calculate_material_consumption(allocations):
    """
    Calculate total slab length consumed by the allocations.
    """
    return sum(
        allocation.required_length
        for allocation in allocations
    )



def calculate_yield_loss(allocations):
    """
    Calculate total unused length on slabs that are actually used.
    """

    consumption_by_slab = {}

    for allocation in allocations:

        slab_id = allocation.slab.slab_id

        consumption_by_slab[slab_id] = (
            consumption_by_slab.get(slab_id, 0)
            + allocation.required_length
        )

    total_yield_loss = 0.0

    for slab_id, used_length in consumption_by_slab.items():

        slab = next(
            allocation.slab
            for allocation in allocations
            if allocation.slab.slab_id == slab_id
        )

        if used_length > slab.length:
            raise ValueError(
                f"Capacity exceeded for slab {slab_id}: "
                f"required {used_length:.2f} m, "
                f"available {slab.length:.2f} m"
            )

        total_yield_loss += (
            slab.length - used_length
        )

    return total_yield_loss


def calculate_slab_utilisation(slabs, allocations):
    """
    Calculate utilisation of slabs that are used in the allocation.

    Returns utilisation as a percentage.
    """

    consumption_by_slab = {}

    for allocation in allocations:

        slab_id = allocation.slab.slab_id

        consumption_by_slab[slab_id] = (
            consumption_by_slab.get(slab_id, 0.0)
            + allocation.required_length
        )

    total_capacity = 0.0
    total_consumption = 0.0

    for slab in slabs:

        if slab.slab_id in consumption_by_slab:

            total_capacity += slab.length
            total_consumption += (
                consumption_by_slab[slab.slab_id]
            )

    if total_capacity == 0:
        return 0.0

    return (
        total_consumption /
        total_capacity
    ) * 100

### Stage-2 : Data loading & initialisation of orders and allocations

In [69]:
data_dir =  r"D:\Projects\slab-allocation-system\intelligent-slab-allocation-system\data"

In [176]:
import os
import pandas as pd

#slab_file = os.path.join(data_dir, "slab_inventory_input.xlsx")
slab_file = os.path.join(data_dir, "slab_inventory_input_s30_p50.xlsx")
#order_file = os.path.join(data_dir, "orders_input.xlsx")
order_file = os.path.join(data_dir, "orders_input_s30_p50.xlsx")

slab_df = pd.read_excel(
    slab_file,
    sheet_name="Slab_Inventory"
)

order_df = pd.read_excel(
    order_file,
    sheet_name="Orders"
)

In [177]:
slabs = []

for _, row in slab_df.iterrows():

    slab = Slab(
        slab_id=row["slab_id"],
        length=row["length"],
        grade=row["grade"],
        quality=row["quality"],
        supplier=row["supplier"],
        surface_class=row["surface_class"],
        status=row["status"]
    )

    slabs.append(slab)

In [178]:
orders = []

for order_id, group in order_df.groupby("order_id"):

    first_row = group.iloc[0]

    plates = []

    for _, row in group.iterrows():

        plate = Plate(
            plate_id=row["plate_id"],
            order_id=row["order_id"],
            length=row["length"],
            width=row["width"],
            thickness=row["thickness"],
            grade=row["grade"],
            quality=row["quality"]
        )

        plates.append(plate)

    allowed_suppliers = first_row["allowed_suppliers"]

    if pd.notna(allowed_suppliers):
        allowed_suppliers = [
            x.strip()
            for x in str(allowed_suppliers).split(",")
        ]
    else:
        allowed_suppliers = None

    required_surface_class = (
        first_row["required_surface_class"]
        if pd.notna(first_row["required_surface_class"])
        else None
    )

    order = Order(
        order_id=order_id,
        customer=first_row["customer"],
        plates=plates,
        allowed_suppliers=allowed_suppliers,
        required_surface_class=required_surface_class,
        require_full_fulfilment=bool(
            first_row["require_full_fulfilment"]
        )
    )

    orders.append(order)

In [179]:
print(f"Number of slabs: {len(slabs)}")
print(f"Number of orders: {len(orders)}")
print(
    f"Number of plates: "
    f"{sum(len(order.plates) for order in orders)}"
)

Number of slabs: 30
Number of orders: 15
Number of plates: 50


#### testing the loaded and initialised data

In [180]:
slab = slabs[3]          # S001, G1/Q1
order = orders[1]        # O001
plate = order.plates[0] # P001, G1/Q1

compatible, reason = is_compatible(
    slab,
    plate,
    order
)

print("Compatible:", compatible)
print("Reason:", reason)

Compatible: False
Reason: Quality specification mismatch


### Stage-3: Allocation with a heuristic algorithm

``Greedy Baseline Heuristic``: We built a baseline heuristic that sequentially allocates each plate to the feasible compatible slab with the smallest remaining capacity after allocation. The algorithm also respects compatibility, slab capacity, and order fulfilment constraints.

For each order, plates are considered sequentially. For each plate, compatible slabs with sufficient remaining capacity are identified. Then, slab leaving the smallest remaining capacity after allocation is selected (best-fit rule).

An order is committed only if all of its plates can be allocated. If any plate cannot be allocated, the temporary allocations for  the entire order are discarded.

In [182]:
def get_slab_consumption(allocations):
    """
    Return total allocated length for each slab.
    """

    consumption = {}

    for allocation in allocations:

        slab_id = allocation.slab.slab_id

        consumption[slab_id] = (
            consumption.get(slab_id, 0.0)
            + allocation.required_length
        )

    return consumption

In [183]:
consumption = get_slab_consumption(allocations)

print(consumption)

{'S002': 16.555851063829785}


In [184]:
def greedy_heuristic_allocate(slabs, orders):
    """
    Perform greedy allocation of slabs to plates in orders.

    Returns
    -------
    allocations : list
        Feasible slab-to-plate Allocation objects.

    unallocated_orders : dict
        Orders that could not be fully allocated, together with the
        reason for failure.
    """

    allocations = []
    unallocated_orders = {}

    # Process orders sequentially
    for order in orders:

        # Temporary allocations for the current order.
        # These are committed only if the complete order can be fulfilled.
        temporary_allocations = []

        # Process plates within the current order
        for plate in order.plates:

            feasible_candidates = []

            # Examine all slabs as potential candidates
            for slab in slabs:

                # Check grade, quality, customer restrictions,
                # and inventory status.
                compatible, _ = is_compatible(
                    slab,
                    plate,
                    order
                )

                if not compatible:
                    continue

                # Calculate material required from this slab
                required_length = calculate_required_length(
                    slab,
                    plate
                )

                # Material already committed to this slab
                committed_usage = sum(
                    allocation.required_length
                    for allocation in allocations
                    if allocation.slab.slab_id == slab.slab_id
                )

                # Material temporarily assigned to this slab
                # within the current order
                temporary_usage = sum(
                    allocation.required_length
                    for allocation in temporary_allocations
                    if allocation.slab.slab_id == slab.slab_id
                )

                used_length = (
                    committed_usage +
                    temporary_usage
                )

                # Remaining slab capacity
                remaining_capacity = (
                    slab.length - used_length
                )

                # Check slab capacity constraint
                if required_length <= remaining_capacity:

                    remaining_after_allocation = (
                        remaining_capacity - required_length
                    )

                    feasible_candidates.append(
                        (
                            remaining_after_allocation,
                            slab
                        )
                    )

            # -------------------------------------------------
            # No feasible slab for this plate
            # -------------------------------------------------

            if not feasible_candidates:

                unallocated_orders[order.order_id] = (
                    f"Could not allocate plate "
                    f"{plate.plate_id}: "
                    f"no compatible slab with sufficient "
                    f"remaining capacity"
                )

                # Entire order must be rejected
                temporary_allocations = []
                break

            # -------------------------------------------------
            # Best-fit greedy selection
            # -------------------------------------------------

            feasible_candidates.sort(
                key=lambda candidate: candidate[0]
            )

            _, selected_slab = feasible_candidates[0]

            temporary_allocations.append(
                Allocation(
                    selected_slab,
                    plate,
                    order
                )
            )

        # -----------------------------------------------------
        # Commit the order only if every plate was allocated
        # -----------------------------------------------------

        if len(temporary_allocations) == len(order.plates):

            allocations.extend(
                temporary_allocations
            )

    return allocations, unallocated_orders

In [185]:
def calculate_kpis(
    slabs,
    orders,
    allocations,
    unallocated_orders
):

    material_consumption = calculate_material_consumption(
        allocations
    )

    yield_loss = calculate_yield_loss(
        allocations
    )

    utilisation = calculate_slab_utilisation(
        slabs,
        allocations
    )


    return {
        "material_consumption": material_consumption,
        "yield_loss": yield_loss,
        "slab_utilisation": utilisation,
        "order_fulfilment": (len(orders) - len(unallocated_orders)) / len(orders) if len(orders) > 0 else 0.0
    }

In [186]:
greedy_allocations, unallocated_orders = greedy_heuristic_allocate(slabs, orders)


In [187]:
for allocation in greedy_allocations:

    print(
        allocation.plate.plate_id,
        "->",
        allocation.slab.slab_id,
        "|",
        allocation.required_length
    )

P001 -> S025 | 11.968085106382977
P002 -> S003 | 4.587765957446808
P003 -> S018 | 12.287234042553191
P004 -> S002 | 9.047872340425531
P005 -> S005 | 6.787234042553191
P006 -> S019 | 4.654255319148937
P007 -> S005 | 3.5106382978723403
P008 -> S008 | 8.329787234042552
P009 -> S019 | 3.7234042553191493
P010 -> S012 | 7.210227272727273
P011 -> S021 | 3.681818181818182
P012 -> S009 | 8.25
P013 -> S003 | 3.1914893617021276
P014 -> S017 | 7.340425531914894
P015 -> S001 | 7.047872340425532
P019 -> S008 | 3.351063829787234
P020 -> S007 | 4.452127659574468
P021 -> S007 | 5.531914893617022
P022 -> S006 | 3.1914893617021276
P023 -> S006 | 8.457446808510639
P024 -> S020 | 3.5904255319148937
P025 -> S010 | 4.438920454545454
P026 -> S022 | 8.735795454545453
P027 -> S022 | 2.352272727272727
P028 -> S023 | 6.136363636363637
P029 -> S013 | 4.261363636363636
P030 -> S013 | 6.40625
P034 -> S015 | 6.25
P035 -> S023 | 3.579545454545454
P036 -> S026 | 5.859375


In [188]:
unallocated_orders

{'O005': 'Could not allocate plate P017: no compatible slab with sufficient remaining capacity',
 'O010': 'Could not allocate plate P033: no compatible slab with sufficient remaining capacity',
 'O012': 'Could not allocate plate P038: no compatible slab with sufficient remaining capacity',
 'O013': 'Could not allocate plate P043: no compatible slab with sufficient remaining capacity',
 'O014': 'Could not allocate plate P047: no compatible slab with sufficient remaining capacity',
 'O015': 'Could not allocate plate P050: no compatible slab with sufficient remaining capacity'}

In [189]:
greedy_kpis = calculate_kpis(
    slabs,
    orders,
    greedy_allocations,
    unallocated_orders
)

print(greedy_kpis)

{'material_consumption': 178.21246373307542, 'yield_loss': 61.78753626692456, 'slab_utilisation': 74.25519322211477, 'order_fulfilment': 0.6}


### Stage-4: Allocation with Optimisation Modelling

In [190]:
from pyscipopt import Model, quicksum

In [ ]:
# =========================================================
# Step 1: Import SCIP optimisation tools
# =========================================================
#
# Model:
#     Creates the SCIP mixed-integer optimisation model.
#
# quicksum:
#     Efficiently builds linear expressions involving
#     multiple decision variables.
# =========================================================

from pyscipopt import Model, quicksum

opti_model = Model("slab_allocation")

print("SCIP model created")

SCIP model created


In [ ]:
# =========================================================
# Step 2: Flatten orders into individual plates
# =========================================================
#
# The optimisation operates at plate-to-slab level.
# Therefore, we create a single list containing all plates
# while retaining the parent order of each plate.
# =========================================================

plates = []
plate_to_order = {}

for order in orders:

    for plate in order.plates:

        plates.append(plate)

        plate_to_order[plate.plate_id] = order

In [192]:
print("Orders:", len(orders))
print("Plates:", len(plates))

Orders: 15
Plates: 50


In [193]:
print(
    plates[0].plate_id,
    plate_to_order[plates[0].plate_id].order_id
)

P001 O001


In [ ]:

# =========================================================
# Step 3: Precompute feasible plate-slab combinations
# =========================================================
#
# slab-plate combination constraint matching, before the optimisation is run, 
# to reduce the search space.
#
# slab-plate combination constraint matching, before the optimisation is run, 
    # to reduce the search space.
#
# Required lengths are calculated once here rather than
# repeatedly during optimisation.
# =========================================================

required_lengths = {}
feasible_pairs = {}

for plate in plates:

    plate_id = plate.plate_id
    order = plate_to_order[plate_id]

    feasible_pairs[plate_id] = []

    for slab in slabs:

        compatible, _ = is_compatible(
            slab,
            plate,
            order
        )

        if not compatible:
            continue

        required_length = calculate_required_length(
            slab,
            plate
        )

        if required_length > slab.length:
            continue

        required_lengths[
            plate_id,
            slab.slab_id
        ] = required_length

        feasible_pairs[
            plate_id
        ].append(slab.slab_id)

In [196]:
print(
    "Total feasible pairs:",
    len(required_lengths)
)

Total feasible pairs: 242


In [ ]:
# =========================================================
# Step 4: Check for plates with no feasible slab
# =========================================================
#
# A plate with no feasible slab cannot currently be
# produced from the available inventory.
#
# We check this before constructing the optimisation model
# so that inventory/compatibility issues can be identified
# explicitly.
# =========================================================

plates_without_candidates = [
    plate_id
    for plate_id, candidates
    in feasible_pairs.items()
    if len(candidates) == 0
]

print(
    "Plates with no feasible slab:",
    plates_without_candidates
)

Plates with no feasible slab: []


In [ ]:


# =========================================================
# Step 5: Create plate-to-slab decision variables
# =========================================================
#
# x[p,s] = 1:
#     Plate p is allocated to slab s.
#
# x[p,s] = 0:
#     Plate p is not allocated to slab s.
#
# Variables are created only for feasible plate-slab pairs.
# =========================================================

x = {}

for plate_id, slab_ids in feasible_pairs.items():

    for slab_id in slab_ids:

        x[
            plate_id,
            slab_id
        ] = opti_model.addVar(
            vtype="B",
            name=f"x_{plate_id}_{slab_id}"
        )

print("Allocation variables:", len(x))

Allocation variables: 242


In [ ]:
# =========================================================
# Step 6: Create order fulfilment variables
# =========================================================
#
# z[o] = 1:
#     The complete order is fulfilled.
#
# z[o] = 0:
#     The order is not fulfilled.
#
# This allows the model to represent inventory shortages
# while preventing partial fulfilment of an order.
# =========================================================

z = {}

for order in orders:

    z[order.order_id] = opti_model.addVar(
        vtype="B",
        name=f"z_{order.order_id}_fulfilled"
    )

print("Order variables:", len(z))

Order variables: 15


In [ ]:
# =========================================================
# Step 7: Enforce complete order fulfilment
# =========================================================
#
# For every plate belonging to an order:
#
#     sum(x[p,s]) = z[o]
#
# If z[o] = 1:
#     every plate in the order must be allocated exactly once.
#
# If z[o] = 0:
#     none of the plates in the order can be allocated.
#
# Therefore, an order cannot be partially fulfilled.
# =========================================================

for order in orders:

    order_id = order.order_id

    for plate in order.plates:

        plate_id = plate.plate_id

        opti_model.addCons(
            quicksum(
                x[plate_id, slab_id]
                for slab_id in feasible_pairs[plate_id]
            )
            == z[order_id],

            name=f"Order_{order_id}_Plate_{plate_id}"
        )

In [ ]:
# =========================================================
# Step 8: Enforce slab capacity constraints
# =========================================================
#
# The total material required by all plates assigned to a
# slab cannot exceed the physical length of that slab.
#
#     sum(required_length[p,s] * x[p,s]) <= slab.length
#
# This prevents multiple plates from collectively consuming
# more material than the selected slab can provide.
# =========================================================

for slab in slabs:

    slab_id = slab.slab_id

    opti_model.addCons(
        quicksum(
            required_lengths[plate_id, slab_id]
            * x[plate_id, slab_id]

            for plate_id, sid in x
            if sid == slab_id
        )
        <= slab.length,

        name=f"Slab_{slab_id}_Capacity"
    )

In [ ]:
# =========================================================
# Step 9: Create slab usage variables
# =========================================================
#
# y[s] = 1:
#     Slab s is used by at least one allocation.
#
# y[s] = 0:
#     Slab s is not used.
#
# These variables are required to calculate yield loss,
# because unused length is counted only for slabs that are
# actually selected for production.
# =========================================================

y = {}

for slab in slabs:

    y[slab.slab_id] = opti_model.addVar(
        vtype="B",
        name=f"y_{slab.slab_id}_used"
    )


In [210]:
# =========================================================
# Step 10: Link plate allocation to slab usage
# =========================================================
#
# x[p,s] = 1 means plate p is allocated to slab s
# y[s]   = 1 means slab s is used
#
# If a plate is allocated to a slab, that slab must be used.
# Therefore:
#
#       x[p,s] <= y[s]
#
# This prevents an allocation being made to a slab that
# is not included in the solution.
# =========================================================

for plate_id, slab_id in x:

    opti_model.addCons(
        x[plate_id, slab_id] <= y[slab_id],
        name=f"Link_{plate_id}_{slab_id}"
    )

print("Allocation-to-slab usage constraints added.")

Allocation-to-slab usage constraints added.


In [212]:
# =========================================================
# Step 11: Calculate total material consumption
# =========================================================
#
# required_lengths[p,s] contains the slab length required
# when plate p is produced from slab s.
#
# The expression below calculates the total slab material
# consumed by the selected allocations.
# =========================================================

total_material_consumption = quicksum(
    required_lengths[plate_id, slab_id]
    * x[plate_id, slab_id]

    for plate_id, slab_id in x
)

print("Material consumption expression created.")

Material consumption expression created.


In [213]:
# =========================================================
# Step 12: Calculate total slab length used
# =========================================================
#
# y[s] = 1 indicates that slab s is used.
#
# Therefore, the total length of slabs used is:
#
#       sum(length[s] * y[s])
# =========================================================

total_slab_length_used = quicksum(
    slab.length * y[slab.slab_id]
    for slab in slabs
)

print("Total slab length expression created.")

Total slab length expression created.


In [214]:
# =========================================================
# Step 13: Define total yield loss
# =========================================================
#
# Yield loss represents the unused length of slabs that
# are opened/used for production.
#
#       yield loss =
#       slab length used - material consumed
# =========================================================

total_yield_loss = (
    total_slab_length_used
    - total_material_consumption
)

print("Yield loss expression created.")

Yield loss expression created.


In [ ]:
# =========================================================
# Step 14: Define order fulfilment reward
# =========================================================
#
# z[o] = 1 means the complete order is fulfilled.
#
# We first want to maximise the number of fulfilled orders.
# A sufficiently large value M is therefore assigned to
# each fulfilled order.
# =========================================================
# maximizing 


total_orders_fulfilled = quicksum(
    z[order.order_id]
    for order in orders
)

# order fulfilment priority is set to the total length of all slabs, 
# which is a sufficiently large value to prioritise order fulfilment 
# over yield loss minimisation.
order_fulfilment_priority = sum(
    slab.length
    for slab in slabs
)

print("Order fulfilment expression created.")


Order fulfilment expression created.
Fulfilment priority M: 370


In [ ]:
# =========================================================
# Step 15: Define optimisation objective
# =========================================================
#
# Primary objective:
#     maximise the number of completely fulfilled orders.
#
# Secondary objective:
#     minimise yield loss.
#
# The large coefficient M gives order fulfilment priority
# over yield loss.
# =========================================================

objective = (
    order_fulfilment_priority * total_orders_fulfilled 
    - total_yield_loss
)

opti_model.setObjective(
    objective,
    "maximize"
)

print("SCIP objective defined.")

SCIP objective defined.


In [217]:
# =========================================================
# Step 16: Inspect optimisation model
# =========================================================

print(
    "Number of allocation variables (x):",
    len(x)
)

print(
    "Number of order fulfilment variables (z):",
    len(z)
)

print(
    "Number of slab usage variables (y):",
    len(y)
)

print(
    "Number of constraints:",
    opti_model.getNConss()
)

Number of allocation variables (x): 242
Number of order fulfilment variables (z): 15
Number of slab usage variables (y): 30
Number of constraints: 564


In [221]:
# =========================================================
# Step 17: Solve the slab allocation model
# =========================================================
import time

time_limit = 30  # seconds

opti_model.setRealParam(
    "limits/time",
    time_limit
)
start_time = time.time()
opti_model.optimize()
end_time = time.time()
print(
    "SCIP status:",
    opti_model.getStatus()
)
print(
    "Optimization time:",
    end_time - start_time
)

SCIP status: optimal
Optimization time: 6.961822509765625e-05


In [222]:
# Objective value of the best solution found
if opti_model.getNSols() > 0:

    print(
        "Objective value:",
        opti_model.getObjVal()
    )

else:

    print("No feasible solution found.")

Objective value: 4063.8152200193417


In [223]:
# =========================================================
# Step 18: Extract SCIP allocation solution
# =========================================================
#
# Convert the selected binary x[p,s] variables back into
# the same Allocation objects used by the greedy heuristic.
#
# This allows both approaches to use exactly the same
# KPI calculation functions.
# =========================================================

scip_allocations = []

if opti_model.getNSols() > 0:

    best_solution = opti_model.getBestSol()

    for plate_id, slab_id in x:

        value = opti_model.getSolVal(
            best_solution,
            x[plate_id, slab_id]
        )

        # Binary variable selected by SCIP
        if value > 0.5:

            plate = next(
                plate
                for plate in plates
                if plate.plate_id == plate_id
            )

            slab = next(
                slab
                for slab in slabs
                if slab.slab_id == slab_id
            )

            order = plate_to_order[plate_id]

            scip_allocations.append(
                Allocation(
                    slab,
                    plate,
                    order
                )
            )


print(
    "SCIP allocations:",
    len(scip_allocations)
)

SCIP allocations: 35


In [224]:
# =========================================================
# Step 19: Identify unfulfilled orders
# =========================================================

scip_allocated_plate_ids = {
    allocation.plate.plate_id
    for allocation in scip_allocations
}

scip_unallocated_orders = {}

for order in orders:

    missing_plates = [
        plate.plate_id
        for plate in order.plates
        if plate.plate_id not in scip_allocated_plate_ids
    ]

    if missing_plates:

        scip_unallocated_orders[order.order_id] = (
            f"Could not allocate plates: {missing_plates}"
        )


print(
    "Unallocated SCIP orders:",
    scip_unallocated_orders
)

Unallocated SCIP orders: {'O001': "Could not allocate plates: ['P001', 'P002', 'P003', 'P004']", 'O002': "Could not allocate plates: ['P005', 'P006', 'P007', 'P008', 'P009']", 'O003': "Could not allocate plates: ['P010', 'P011', 'P012']", 'O005': "Could not allocate plates: ['P016', 'P017', 'P018']"}


In [ ]:
greedy_kpis = calculate_kpis(
    slabs,
    orders,
    greedy_allocations,
    unallocated_orders
)

scip_kpis = calculate_kpis(
    slabs,
    orders,
    scip_allocations,
    scip_unallocated_orders
)


{'material_consumption': 194.81522001934235,
 'yield_loss': 6.184779980657649,
 'slab_utilisation': 96.92299503449868,
 'order_fulfilment': 0.7333333333333333}

In [227]:
print("Greedy KPIs")
print(greedy_kpis)

print("\nSCIP KPIs")
print(scip_kpis)

Greedy KPIs
{'material_consumption': 178.21246373307542, 'yield_loss': 61.78753626692456, 'slab_utilisation': 74.25519322211477, 'order_fulfilment': 0.6}

SCIP KPIs
{'material_consumption': 194.81522001934235, 'yield_loss': 6.184779980657649, 'slab_utilisation': 96.92299503449868, 'order_fulfilment': 0.7333333333333333}


In [228]:
print(
    "Greedy fulfilled orders:",
    len(orders) - len(unallocated_orders)
)

print(
    "SCIP fulfilled orders:",
    len(orders) - len(scip_unallocated_orders)
)

Greedy fulfilled orders: 9
SCIP fulfilled orders: 11


### Appendix

In [198]:
plate_id = "P010"

plate = next(
    plate
    for plate in plates
    if plate.plate_id == plate_id
)

order = plate_to_order[plate_id]

print("Plate:", plate.plate_id)
print("Order:", order.order_id)
print("Grade:", plate.grade)
print("Quality:", plate.quality)
print("Length:", plate.length)
print("Width:", plate.width)
print("Thickness:", plate.thickness)

print("\nCandidate slabs:")

Plate: P010
Order: O003
Grade: G2
Quality: Q1
Length: 4.7
Width: 1.8
Thickness: 0.03

Candidate slabs:


In [199]:
for slab in slabs:

    compatible, reason = is_compatible(
        slab,
        plate,
        order
    )

    if compatible:

        required_length = calculate_required_length(
            slab,
            plate
        )

        print(
            slab.slab_id,
            "| length:", slab.length,
            "| required:", round(required_length, 2),
            "| capacity:",
            "OK" if required_length <= slab.length
            else "TOO SHORT"
        )

    else:

        print(
            slab.slab_id,
            "| INCOMPATIBLE:",
            reason
        )

S001 | INCOMPATIBLE: Steel grade mismatch
S002 | INCOMPATIBLE: Steel grade mismatch
S003 | INCOMPATIBLE: Steel grade mismatch
S004 | INCOMPATIBLE: Steel grade mismatch
S005 | INCOMPATIBLE: Steel grade mismatch
S006 | INCOMPATIBLE: Steel grade mismatch
S007 | INCOMPATIBLE: Steel grade mismatch
S008 | INCOMPATIBLE: Steel grade mismatch
S009 | length: 10 | required: 7.21 | capacity: OK
S010 | length: 13 | required: 7.21 | capacity: OK
S011 | length: 15 | required: 7.21 | capacity: OK
S012 | length: 9 | required: 7.21 | capacity: OK
S013 | INCOMPATIBLE: Quality specification mismatch
S014 | INCOMPATIBLE: Quality specification mismatch
S015 | INCOMPATIBLE: Quality specification mismatch
S016 | INCOMPATIBLE: Quality specification mismatch
S017 | INCOMPATIBLE: Steel grade mismatch
S018 | INCOMPATIBLE: Steel grade mismatch
S019 | INCOMPATIBLE: Steel grade mismatch
S020 | INCOMPATIBLE: Steel grade mismatch
S021 | length: 9 | required: 7.21 | capacity: OK
S022 | length: 14 | required: 7.21 | cap

In [201]:
print("Number of slabs:", len(slab_df))

print("\nStatus:")
print(slab_df["status"].value_counts())

print("\nGrade / Quality:")
print(
    pd.crosstab(
        slab_df["grade"],
        slab_df["quality"]
    )
)

print("\nSuppliers:")
print(slab_df["supplier"].value_counts())

Number of slabs: 30

Status:
status
Available    26
Reserved      2
Incoming      2
Name: count, dtype: int64

Grade / Quality:
quality  Q1  Q2
grade          
G1        8   7
G2        7   8

Suppliers:
supplier
SUP1    15
SUP2    15
Name: count, dtype: int64
